# **EEC 174AY Lab 2: Supervised Learning**


## **Outline**

1. Basics of Python Pandas for Reading & Manipulating Data
    - Task #1
2. Introduction to Scikit-Learn (package used for more traditional machine learning tools), create an RF model
    - Task #2
    - Task #3
3. Improving your model: Feature Selection
    - Task #4a
    - Task #4b (Optional)
4. K-Fold Validation
    - Task #5

# **1. Basics of Python Pandas for Reading & Manipulating Data**

### **1.1: Reading Data into Memory**

For this lab, we will be looking at ventilator data of patients' breath patterns. We will be working with *train x* and *train y* datasets to train a model. Then, we will use *test x* and *test y* datasets to validate our model.

First, let's load in our data:

In [ ]:
# make sure we can plot in future if we want
# %matplotlib notebook
# make sure to ignore warnings
import warnings
warnings.simplefilter('ignore')
# Import statement for pandas
import pandas as pd
# This is just a small configuration change for purposes of the class
pd.options.display.max_rows = 10

# Get our train X and y datasets for the problem
train_x = pd.read_csv('ece174_pva_train_x.csv')
train_y = pd.read_csv('ece174_pva_train_y.csv')

# Get our validation X and y datasets for the problem.
test_x = pd.read_csv('ece174_pva_validation_x.csv')
test_y = pd.read_csv('ece174_pva_validation_y.csv')

In [ ]:
# output some rows of the dataset just to get a better feel for the information
train_x

**Features**

The data currently has 10 columns above. Later in Assignment 1 you will increase these to 18. Below is a description of all 18 columns or *features*. Note that some column information cannot be featurized such as `patient` or `breath_id`.

 * `breath_id`: matches with a specific breath identifier from the raw data file.
 * `patient`: the patient the data came from
 * `min_flow`: The minimum flow observation on the breath
 * `max_flow`: The maximum flow observation on the breath
 * `tvi`: The inhaled volume of air for each breath
 * `tve`: The exhaled volume of air for each breath
 * `tve_tvi_ratio`: The ratio of `tve / tvi`
 * `i_time`: The amount of time patient was breathing in for each breath
 * `e_time`: The amount of time patient was breathing out for each breath
 * `ie_ratio`: The ratio of `i_time / e_time`
 * `rr`: The respiratory rate in number of breaths per minute. Measured by `60 / (i_time + e_time)`
 * `min_pressure`: the minimum pressure observation on the breath
 * `max_pressure`: the maximum pressure observation on the breath
 * `peep`: - the baseline pressure setting on the ventilator
 * `pip`: the maximum pressure setting of inspiration. Slight difference compared to max_pressure
 * `maw`: the mean pressure for the entire breath
 * `ip_auc`: the area under the curve of the inspiratory pressure
 * `ep_auc`: the area under the curve of the expiratory pressure

### **1.2: Featurization**

Featurization is the process where you extract information from raw data. This information can then be fed into a machine learning algorithm to perform the task you want. In the current case we will need to extract additional information from the ventilator data in order to create a valid machine learning classifier.

#### Processing the Data
The first step we need to do is to be able to read the raw data files and put them into memory.

Here, we load in the ventilator data stored in the patient CSV files and organize them by breath cycles.

In [ ]:
import csv


def process_ventilator_data(filename):
    descriptor = open(filename)
    reader = csv.reader(descriptor)
    breath_id = 1

    all_breath_data = []
    current_flow_data = []
    current_pressure_data = []

    for row in reader:
        if (row[0].strip() == 'BS' or row[0].strip() == 'BE') and current_flow_data != []:
            all_breath_data.append({'breath_id': breath_id, 'flow': current_flow_data, 'pressure': current_pressure_data})
            breath_id += 1
            current_flow_data = []
            current_pressure_data = []
        else:
            try:
                current_flow_data.append(round(float(row[0]), 2))
                current_pressure_data.append(round(float(row[1]), 2))
            except (IndexError, ValueError):
                continue
    return all_breath_data

Now that we have the capability to read ventilator data into memory, we should try to visualize what the data looks like.
We will be extracting just one breath cycle to visualize the breath flow and pressure of the patient.

In [ ]:
import os
from glob import glob

import matplotlib.pyplot as plt

data_files = glob(os.path.join('data', '*/*.csv'))
example_file = data_files[0]
breath_data = process_ventilator_data(example_file)
flow = breath_data[0]['flow']
pressure = breath_data[0]['pressure']
plt.plot(flow, label='flow')
plt.plot(pressure, label='pressure')
plt.legend()
plt.show()

## **Task \#1: Featurization** (35 points)

Now that we've visualized the data we need to featurize the data so we can use it in a ML algorithm. We'll need a bit more code to do this. We've already given you 10 features above that you can use for the current models. Your assignment will be to process the rest of the 8 features based on the requirements and hints that I've given in the code below.

In [ ]:

import pandas as pd

train_x = pd.read_csv("ece174_pva_train_x.csv")
test_x = pd.read_csv("ece174_pva_validation_x.csv")

print("Training shape:", train_x.shape)
print("Training columns:", train_x.columns.tolist())

print("Validation shape:", test_x.shape)
print("Validation columns:", test_x.columns.tolist())


In [ ]:

import os
import csv
import numpy as np
import pandas as pd
from glob import glob
from statistics import mean
from scipy.integrate import simpson


In [ ]:

def process_ventilator_data(filename):
    all_breath_data = []
    current_flow_data = []
    current_pressure_data = []
    breath_id = 1

    with open(filename, newline="") as descriptor:
        reader = csv.reader(descriptor)

        for row in reader:
            if (row[0].strip() in ("BS", "BE")
                    and current_flow_data):
                all_breath_data.append({
                    "breath_id": breath_id,
                    "flow": current_flow_data,
                    "pressure": current_pressure_data
                })

                breath_id += 1
                current_flow_data = []
                current_pressure_data = []

            else:
                try:
                    current_flow_data.append(round(float(row[0]), 2))
                    current_pressure_data.append(round(float(row[1]), 2))
                except (IndexError, ValueError):
                    continue

    return all_breath_data


In [ ]:

def extract_features_for_file(filename, existing_features):
    patient = os.path.basename(os.path.dirname(filename))
    all_breath_data = process_ventilator_data(filename)
    all_features = []

    for breath_data in all_breath_data:
        breath_id = breath_data["breath_id"]

        matching = existing_features[
            existing_features["breath_id"] == breath_id
        ]

        if matching.empty:
            continue

        existing_row = matching.iloc[0]

        i_time = float(existing_row["i_time"])
        tve = float(existing_row["tve"])
        max_flow = existing_row["max_flow"]
        min_flow = existing_row["min_flow"]
        max_pressure = existing_row["max_pressure"]
        peep = existing_row["peep"]
        ip_auc = existing_row["ip_auc"]
        ep_auc = existing_row["ep_auc"]

        flow = np.asarray(breath_data["flow"], dtype=float)
        pressure = np.asarray(breath_data["pressure"], dtype=float)

        if len(flow) == 0 or len(pressure) == 0:
            continue

        x0_index = max(1, min(int(i_time / 0.02), len(flow) - 1))

        # 1. Expiratory time (seconds)
        e_time = (len(flow) - x0_index) * 0.02

        # 2. Inspiratory-to-expiratory time ratio
        i_e_ratio = i_time / e_time if e_time > 0 else np.nan

        # 3. Respiratory rate (breaths per minute)
        total_time = i_time + e_time
        rr = 60 / total_time if total_time > 0 else np.nan

        # 4. Tidal volume inhaled (mL)
        tvi = simpson(flow[:x0_index], dx=0.02) * 1000 / 60

        # 5. Exhaled-to-inhaled tidal volume ratio
        tve_tvi_ratio = tve / tvi if tvi != 0 else np.nan

        # 6. Minimum pressure during the breath
        min_pressure = float(np.min(pressure))

        # 7. Peak inspiratory pressure
        pip = float(np.max(pressure[:x0_index]))

        # 8. Mean airway pressure over the breath
        maw = float(np.mean(pressure))

        all_features.append([
            breath_id, i_time, e_time, i_e_ratio, rr,
            tvi, tve, tve_tvi_ratio, max_flow, min_flow,
            max_pressure, min_pressure, pip, maw,
            peep, ip_auc, ep_auc, int(patient)
        ])

    columns = [
        "breath_id", "i_time", "e_time", "i_e_ratio", "rr",
        "tvi", "tve", "tve_tvi_ratio", "max_flow", "min_flow",
        "max_pressure", "min_pressure", "pip", "maw",
        "peep", "ip_auc", "ep_auc", "patient"
    ]

    return all_features, columns


In [ ]:

def remake_dataset(dataset):
    data_files = glob(os.path.join("data", "*", "*.csv"))

    patient_to_file_map = {
        os.path.basename(os.path.dirname(filename)): filename
        for filename in data_files
    }

    data = []
    columns = None

    for patient in dataset["patient"].unique():
        patient_id = str(int(patient))

        if patient_id not in patient_to_file_map:
            raise FileNotFoundError(
                f"No waveform CSV found for patient {patient_id}"
            )

        existing_features = dataset[
            dataset["patient"].astype(str) == patient_id
        ]

        breath_data, columns = extract_features_for_file(
            patient_to_file_map[patient_id],
            existing_features
        )

        data.extend(breath_data)

    return pd.DataFrame(data, columns=columns)


In [ ]:

# Keep copies of the original input data
train_x_original = pd.read_csv("ece174_pva_train_x.csv")
test_x_original = pd.read_csv("ece174_pva_validation_x.csv")

# Rebuild the feature datasets
train_x = remake_dataset(train_x_original)
test_x = remake_dataset(test_x_original)

print("Training shape:", train_x.shape)
print("Validation shape:", test_x.shape)

assert train_x.shape[1] == 18, "Expected 18 training columns"
assert test_x.shape[1] == 18, "Expected 18 validation columns"

print("Task #1 checks completed.")


In [ ]:
print("Original training rows:", len(train_x_original))
print("Rebuilt training rows:", len(train_x))

print("Original validation rows:", len(test_x_original))
print("Rebuilt validation rows:", len(test_x))

print("\nMissing values in training:")
print(train_x.isnull().sum())

print("\nMissing values in validation:")
print(test_x.isnull().sum())

In [ ]:
print(train_x[
    ["i_time", "e_time", "i_e_ratio", "rr", "tvi",
     "tve", "tve_tvi_ratio", "min_pressure", "pip", "maw"]
].describe())

print("\nInvalid or infinite values:")
print(
    (~train_x.select_dtypes(include="number").apply(
        lambda col: col.map(lambda x: abs(x) != float("inf"))
    )).sum()
)

In [ ]:
# Inspect breaths with suspicious feature values
suspicious = train_x[
    (train_x["tvi"] <= 0) |
    (train_x["tve"] < 0) |
    (train_x["i_e_ratio"] > 5) |
    (train_x["rr"] > 100) |
    (train_x["pip"] < 0)
]

print("Suspicious breaths:", len(suspicious))

display(
    suspicious[
        ["breath_id", "patient", "i_time", "e_time",
         "i_e_ratio", "rr", "tvi", "tve",
         "tve_tvi_ratio", "pip"]
    ].head(20)
)

In [ ]:
import inspect

print("=== extract_features_for_file ===")
print(inspect.getsource(extract_features_for_file))

print("\n=== process_ventilator_data ===")
print(inspect.getsource(process_ventilator_data))

In [ ]:
import inspect

source = inspect.getsource(extract_features_for_file)
print(source)

# **2. Introduction to Scikit-Learn**

### **2.1: Create Ground Truth**

We will use the corresponding *test y* dataset containing healthy non-PVA, Breath Stacking Asynchrony (bsa), and Double Trigger Asynchrony (dta) labels as the ground truth for training our model.

In [ ]:
# Read the test dataset and set it up.
test_y

What does this mean?

We have 6 columns here
 * `breath_id`: matches with a specific breath identifier from the raw data file.
 * `patient`: the patient the data came from
 * `bsa`: Breath Stacking Asynchrony. A single breath where the patient is trapping air in their chest
 * `dta`: Double Trigger Asynchrony. Two breaths in a row where the patient is trapping air
 * `cough`: What it sounds like, when a patient coughs
 * `suction`:  Nurses perform suction procedures to remove excess fluid from an endotracheal tube. This waveform is indicative of that.

 Each patient will either have healthy breath pattern, breath stacking asynchrony, or double trigger asynchrony, indicated by which label is 1 or 0.

Now that we understand what our columns are, we need to put it into a format where the machine can understand it and create a learning model. Because this is a multiclass model, let's just have *non-PVA breaths* be class 0, *breath stacking* can be class 1, *double trigger* can be class 2.

In [ ]:
# Create a multi-class y vector that we can use for training purposes.
train_y_vector = train_y.bsa * 1 + train_y.dta * 2
test_y_vector = test_y.bsa * 1 + test_y.dta * 2
test_y_vector

Therefore, there will be only three categories in our dataset (non-PVA, bsa, dta).

### **2.2: Create and train a Random Forest Classifier**

Let's finalize the dataset and remove misannotated examples first.

In [ ]:
# See if there places where the data was mis-annotated, where both double trigger and breath stack was annotated.
# It's just good to know if this is happening or not so that we can either drop the data, or change it later on.
train_y_vector[train_y_vector > 2]

In [ ]:
# Mark places where data is double annotated.
misannotated_train = train_y_vector > 2
misannotated_test = test_y_vector > 2

# Drop data flagged above (~ is the NOT operator)
train_x = train_x.loc[~misannotated_train]
train_y_vector = train_y_vector.loc[~misannotated_train]

# do same thing for test
test_x = test_x.loc[~misannotated_test]
test_y_vector = test_y_vector.loc[~misannotated_test]

Let's also remove data that is NaN. This is very important because otherwise your model won't train.

In [ ]:
'''
The .any(axis=1) function basically says, if there are any NaNs in this *ROW* then mark the row as true.
The .any(axis=0) would mark columns as True/False, but this isn't helpful now.
'''

nans_train = train_x.isna().any(axis=1)
nans_test = test_x.isna().any(axis=1)

# now filter them out of the dataset in the same way
train_x = train_x.loc[~nans_train]
train_y_vector = train_y_vector.loc[~nans_train]

test_x = test_x.loc[~nans_test]
test_y_vector = test_y_vector.loc[~nans_test]

Any time we drop things from a data frame or series in pandas it is often helpful to re-index the object.
The index is usually a sequential ordering of the rows like 1, 2, ... n. Sometimes it can be different
but for now we'll just use sequential ordering.

In [ ]:
train_x.index = range(len(train_x))
train_y_vector.index = range(len(train_y_vector))

test_x.index = range(len(test_x))
test_y_vector.index = range(len(test_y_vector))

Now we can train the model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# initialize model
model = RandomForestClassifier()

# don't use patient and breath_id columns
columns_to_use = list(set(train_x.columns).difference(['patient', 'breath_id']))

# fit the model to training
model.fit(train_x[columns_to_use], train_y_vector)

# Now that the model is fitted, evaluate how well it is performing
predictions = model.predict(test_x[columns_to_use])


'''
The optimization below helps re-model the data in the train set slightly so that the first double
trigger breath in training set is marked as a breath stack, and kept last double trigger breath a double trigger.
You don't not need to understand this part of the code.
'''


for idx, pred in enumerate(predictions):
    if pred == 2:
        predictions[idx-1] = 2

To evaluate our model, we can use `sklearn.metrics` from *scikit-learn*, since PyTorch doesn't have a built-in metrics module. The `classification_report()` function provides key stats like precision, recall, and F1 score, making it useful for quickly assessing performance, especially with multiple classes.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_y_vector, predictions))

### **2.3 Scaling for ML**
Scaling data is often essential, especially for neural networks, and can improve performance in models like random forests. Common methods include:

#### **2.3.1: Standardization**
Standardization transforms data by subtracting the mean and dividing by the standard deviation:
$$ \frac{x_f - \mu_f}{\sigma_f} $$
Where $x_f$ is the feature, $\mu_f$ is the mean, and $\sigma_f$ is the standard deviation. Scikit-Learn simplifies this:

```python
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
train_set_scaled = scaler.fit_transform(train_set)
test_set_scaled = scaler.transform(test_set)
```

#### **2.3.2 Min-Max Scaling**
This method scales features to a [0, 1] range:

$$ (x_f - min(x_f)) \div (max(x_f) - min(x_f)) $$

Where the `min` function is just finding the minimum value of a feature vector, and the `max` function is finding the maximum value of a feature vector. You can do this quickly in Scikit-Learn too.

```python
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
train_set_scaled = scaler.fit_transform(train_set)
test_set_scaled = scaler.transform(test_set)
```

#### **2.3.3 Robust Scaler**
The robust scaler removes the median and scales by the interquartile range (IQR), making it effective for data with outliers. It's worth experimenting with different scalers to see what works best for your model.

```python
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
train_set_scaled = scaler.fit_transform(train_set)
test_set_scaled = scaler.transform(test_set)
```

## **Task \#2: Create a working model of a random forest classifier with all the features given.** (20 points)

If you've been able to featurize all your information correctly, let's move onto creating a random forest model for the completely featurized dataset. You **must** use one of the scaling strategies. Run your model 10 times to get performance scores for precision, recall, and f1-score. Average and report the results **per class**.

In [23]:
# Task 2: Random Forest with scaling, evaluated over 10 runs

from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

feature_columns = [
    c for c in train_x.columns
    if c not in ("patient", "breath_id")
]

X_train = train_x[feature_columns].astype(float)
X_test = test_x[feature_columns].astype(float)
y_train = train_y_vector.astype(int)
y_test = test_y_vector.astype(int)

class_names = {0: "Non-PVA", 1: "BSA", 2: "DTA"}
scores_by_class = {
    label: {"precision": [], "recall": [], "f1-score": []}
    for label in class_names
}

for run in range(10):
    rf_pipeline = Pipeline([
        ("scaler", RobustScaler()),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42 + run,
            n_jobs=-1
        ))
    ])

    rf_pipeline.fit(X_train, y_train)
    rf_predictions = rf_pipeline.predict(X_test)

    report = classification_report(
        y_test,
        rf_predictions,
        labels=[0, 1, 2],
        output_dict=True,
        zero_division=0
    )

    for label in class_names:
        for metric in ("precision", "recall", "f1-score"):
            scores_by_class[label][metric].append(report[str(label)][metric])

print("Random Forest: average scores over 10 runs")
for label, name in class_names.items():
    print(f"\\nClass {label} ({name})")
    for metric, values in scores_by_class[label].items():
        print(f"Average {metric}: {np.mean(values):.4f} "
              f"(std: {np.std(values):.4f})")

# Keep a final Random Forest model for later comparisons.
rf_model = Pipeline([
    ("scaler", RobustScaler()),
    ("classifier", RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ))
])
rf_model.fit(X_train, y_train)
rf_test_predictions = rf_model.predict(X_test)

print("\\nClassification report for the final Random Forest model:")
print(classification_report(
    y_test, rf_test_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))


Random Forest: average scores over 10 runs
\nClass 0 (Non-PVA)
Average precision: 0.9004 (std: 0.0017)
Average recall: 0.9625 (std: 0.0013)
Average f1-score: 0.9304 (std: 0.0011)
\nClass 1 (BSA)
Average precision: 0.8444 (std: 0.0028)
Average recall: 0.9661 (std: 0.0013)
Average f1-score: 0.9011 (std: 0.0019)
\nClass 2 (DTA)
Average precision: 1.0000 (std: 0.0000)
Average recall: 0.0250 (std: 0.0168)
Average f1-score: 0.0483 (std: 0.0317)
\nClassification report for the final Random Forest model:
              precision    recall  f1-score   support

     Non-PVA       0.90      0.96      0.93       842
         BSA       0.84      0.97      0.90       301
         DTA       1.00      0.01      0.02       104

    accuracy                           0.88      1247
   macro avg       0.91      0.65      0.62      1247
weighted avg       0.89      0.88      0.85      1247



## **Task #3: Comparison of Supervised ML Algorithms** (10 points)

In machine learning, much focus is placed on choosing the right algorithm. However, trying a different algorithm doesn’t always improve performance and can sometimes complicate things. That said, it's important to explore different models that fit your problem.

In this assignment, you need to implement *at least one* Supervised Learning model and compare its performance with the Random Forest model. Report metrics such as misclassification rate, precision, recall, F1-score, etc.

### **Supervised Learning Algorithms to Consider:**
- Neural Networks (use PyTorch/TensorFlow if preferred)
- Logistic Regression
- Naive Bayes
- SVM (typically for binary classification but can be explored for multi-class)
- etc.

Hints:
### Logistic Regression

Now we’ll run L1-regularized logistic regression on our dataset.
You may use:
```python
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
```
* Fit a L1-regularized logistic regression model by letting python chose the regularization
parameter itself.

Hint:
```python
model = LogisticRegression(penalty='l1') # L1-regularized
```

### kNN Classifier

Use kNN classifier model on our dataset.

```python
from sklearn.neighbors import KNeighborsClassifier
```


In [24]:
# Task 3: Compare Random Forest with L1-regularized Logistic Regression

from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# LogisticRegressionCV selects the regularization strength (C) using cross-validation.
logistic_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegressionCV(
        Cs=10,
        cv=3,
        penalty="l1",
        solver="saga",
        scoring="f1_macro",
        max_iter=5000,
        random_state=42,
        n_jobs=-1
    ))
])

logistic_model.fit(X_train, y_train)
logistic_predictions = logistic_model.predict(X_test)

print("=== Random Forest ===")
print(f"Misclassification rate: {1 - accuracy_score(y_test, rf_test_predictions):.4f}")
print(classification_report(
    y_test, rf_test_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))

print("=== L1 Logistic Regression ===")
print(f"Misclassification rate: {1 - accuracy_score(y_test, logistic_predictions):.4f}")
print(classification_report(
    y_test, logistic_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))

print("Chosen Logistic Regression C values:",
      logistic_model.named_steps["classifier"].C_)


=== Random Forest ===
Misclassification rate: 0.1171
              precision    recall  f1-score   support

     Non-PVA       0.90      0.96      0.93       842
         BSA       0.84      0.97      0.90       301
         DTA       1.00      0.01      0.02       104

    accuracy                           0.88      1247
   macro avg       0.91      0.65      0.62      1247
weighted avg       0.89      0.88      0.85      1247

=== L1 Logistic Regression ===
Misclassification rate: 0.2269
              precision    recall  f1-score   support

     Non-PVA       0.85      0.88      0.86       842
         BSA       0.60      0.74      0.66       301
         DTA       0.50      0.02      0.04       104

    accuracy                           0.77      1247
   macro avg       0.65      0.55      0.52      1247
weighted avg       0.76      0.77      0.75      1247

Chosen Logistic Regression C values: [21.5443469 21.5443469 21.5443469]


# **3. Feature Selection**

We know there are only 3 classes in our labels as shown below.

In [ ]:
test_y

One thing to note is that we are using 16 different features for input into our model. Some of these features can be of little value to classifying whether a breath is asynchronous or not. So, one of the easiest things we can do for ourselves is to reduce the number of features that we have in an intelligent way.

### **3.1: $\chi^2$ Feature Selection** (chi squared)

The chi-squared (χ²) test is an intuitive method for feature selection in classification problems. It evaluates whether a feature is independent of the target variable—if so, the feature may not be useful for the model. A high χ² value with a low p-value (≤ 0.05) suggests that the feature is dependent on the outcome.

Learn more about the chi-squared test on [Wikipedia](https://en.wikipedia.org/wiki/Chi-squared_test) and its application in [feature selection](https://nlp.stanford.edu/IR-book/html/htmledition/feature-selectionchi2-feature-selection-1.html).


[Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.chi2.html) provides a function for performing the χ² test.

In [ ]:
# this is the PrettyPrint function. Just makes things look a bit nicer on output.
from pprint import pprint

from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

# get all columns in our dataset except patient and breath_id
columns_to_use = list(set(train_x.columns).difference(['patient', 'breath_id']))

# must scale feature vectors so they are non-negative
scaler = MinMaxScaler()
train_set = scaler.fit_transform(train_x[columns_to_use])

# the chi2 test will output two things, chi2 and p values. The p values are the most relevant item that we want
# to use. A feature with a p-value between 0 and 0.05 means that a feature might be a good predictor of our outcome.
chi2_vals, pvals = chi2(train_set, train_y_vector)

# mash column names with p-values so we know which p-value belongs to which feature
cols_to_pvals = zip(pvals, columns_to_use)
# Sort the p-values in ascending order (smallest first).
cols_sorted = sorted(cols_to_pvals)
# pretty print the sorted values.
print(cols_sorted)

There are 2 features that had p-values below 0.05:

 * tve
 * ep_auc

So let's use these features for our next model.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler

model = RandomForestClassifier()
columns_to_use = ['tve', 'ep_auc']

scaler = MinMaxScaler()
train_set = scaler.fit_transform(train_x[columns_to_use])
test_set = scaler.transform(test_x[columns_to_use])

model.fit(train_set, train_y_vector)
predictions = model.predict(test_set)
for idx, pred in enumerate(predictions):
    if pred == 2:
        predictions[idx-1] = 2

print(classification_report(test_y_vector, predictions))

Our performance dropped when using the χ² test. Does this mean the method isn't suitable for our problem?

**What's happening?**

Although the χ² test indicates certain features are relevant, they may not generalize well to the test set. This is common in machine learning—features important in training don't always perform well in testing. Are there other feature selection methods that may generalize better?

### **3.2: Expert Feature Selection**

Expert knowledge, such as medical insights, can improve model performance. What medical knowledge can we use here?

#### **3.2.1: Breath Stack (BSA)**
When a patient traps air in their chest, it can be measured by the `tve_tvi_ratio`. Doctors annotated breaths with a `tve_tvi_ratio < .9`, excluding suction/cough or anomalies.

<img src="bsa-breath.png" alt="Drawing" style="width: 400px;"/>

#### **3.2.2: Double Trigger (DTA)**
Double trigger has a double-hump waveform pattern.

<img src="dta-breaths.png" alt="Drawing" style="width: 600px;"/>

Doctors annotated it as:

1. Not an anomaly
2. First breath with `e_time < .3` seconds
3. First breath with `tve_tvi_ratio < .25` or `0.25 <= tve_tvi_ratio < 0.5` and `tve < 100`

## **Task \#4a: Expert Feature Selection** (40 points)

Using the expert features above, write a function to give predictions on the train set.
Then, create a classification report on your predictions.

In [25]:
# Task 4a: Expert-rule predictions on the training set

import numpy as np
from sklearn.metrics import classification_report

def expert_rule_predictions(dataset):
    """
    Apply the expert rules described in the lab:
    - DTA: short expiratory time (< 0.3 s) and a low tve_tvi_ratio,
      using the lab's ratio/tve conditions.
    - BSA: tve_tvi_ratio < 0.9, except breaths already classified as DTA.
    - Otherwise: Non-PVA (class 0).
    """
    required = {"e_time", "tve_tvi_ratio", "tve"}
    missing = required.difference(dataset.columns)
    if missing:
        raise ValueError(f"Missing expert-rule features: {sorted(missing)}")

    e_time = dataset["e_time"].to_numpy(dtype=float)
    ratio = dataset["tve_tvi_ratio"].to_numpy(dtype=float)
    tve = dataset["tve"].to_numpy(dtype=float)

    predictions = np.zeros(len(dataset), dtype=int)

    dta_mask = (
        (e_time < 0.3)
        & (
            (ratio < 0.25)
            | ((ratio >= 0.25) & (ratio < 0.5) & (tve < 100))
        )
    )
    predictions[dta_mask] = 2

    bsa_mask = (ratio < 0.9) & (~dta_mask)
    predictions[bsa_mask] = 1

    return predictions

expert_train_predictions = expert_rule_predictions(train_x)

print("Expert-rule classification report on training data:")
print(classification_report(
    train_y_vector,
    expert_train_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))


Expert-rule classification report on training data:
              precision    recall  f1-score   support

     Non-PVA       0.95      0.95      0.95      4323
         BSA       0.92      0.83      0.87      1412
         DTA       0.00      0.00      0.00       237

    accuracy                           0.88      5972
   macro avg       0.62      0.59      0.61      5972
weighted avg       0.90      0.88      0.89      5972



### **3.3: Other Feature Selection Methods**

You can explore various methods for feature selection. Here’s a brief overview:


#### **3.3.1: Wrapper Methods**
This brute-force approach trains models with all possible feature combinations and selects the best based on performance on the validation set.

**Pros:**
 * Easy to understand and code
 * Works with any model

**Cons:**
 * Prone to overfitting
 * Time-consuming—requires training $n!$ models for $n$ features


#### **3.3.2: PCA (Principal Component Analysis)**
[PCA](https://en.wikipedia.org/wiki/Principal_component_analysis) transforms data into new, uncorrelated features. You choose the number of features, then train and evaluate models.

**Pros:**
 * Speeds up model training through dimensionality reduction
 * Features are linearly uncorrelated
 * Easy to use with libraries like [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html)

**Cons:**
 * Potential loss of information, which can reduce performance
 * Features lose interpretability


#### **3.3.3: Mutual Information**
[Mutual Information](https://en.wikipedia.org/wiki/Mutual_information) measures the dependency between a feature and the target. A higher value indicates greater dependency.

**Pros:**
 * Fast
 * Supported by [Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html)

**Cons:**
 * Like χ², it may not generalize to the test set

#### **3.3.4: Mixed Methods**
Combining methods is common. Start with expert feature selection, then add synthetic methods. Test different approaches, but be cautious of overfitting. Always evaluate using a solid validation set, not the test set.

## **Task \#4b (BONUS, not graded): Finish Expert Feature Selection & Find another Feature Selection Method to Use.**

Finish the coding for expert feature selection and use another feature selection method like PCA/mutual information/wrapper methods for use in your model. Which one performs best?

In [26]:
# Task 4b (bonus): Compare expert rules with mutual-information feature selection

from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

# Evaluate the expert rules on the held-out validation set.
expert_test_predictions = expert_rule_predictions(test_x)
expert_macro_f1 = f1_score(
    test_y_vector, expert_test_predictions,
    average="macro", zero_division=0
)

# Select the 8 most informative features using training data only.
k_features = min(8, len(feature_columns))
mi_model = Pipeline([
    ("scaler", StandardScaler()),
    ("select", SelectKBest(
        score_func=lambda X, y: mutual_info_classif(
            X, y, random_state=42
        ),
        k=k_features
    )),
    ("classifier", RandomForestClassifier(
        n_estimators=200, random_state=42, n_jobs=-1
    ))
])

mi_model.fit(X_train, y_train)
mi_predictions = mi_model.predict(X_test)
mi_macro_f1 = f1_score(
    y_test, mi_predictions, average="macro", zero_division=0
)

selected_mask = mi_model.named_steps["select"].get_support()
selected_features = [
    name for name, selected in zip(feature_columns, selected_mask) if selected
]

print("Features selected by mutual information:")
print(selected_features)

print("\\nExpert rules — validation classification report:")
print(classification_report(
    test_y_vector, expert_test_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))

print("Mutual-information + Random Forest — validation classification report:")
print(classification_report(
    y_test, mi_predictions,
    labels=[0, 1, 2],
    target_names=["Non-PVA", "BSA", "DTA"],
    zero_division=0
))

print(f"Expert-rule macro F1: {expert_macro_f1:.4f}")
print(f"Mutual-information model macro F1: {mi_macro_f1:.4f}")
if mi_macro_f1 > expert_macro_f1:
    print("Better macro F1 on this validation set: mutual-information model.")
elif expert_macro_f1 > mi_macro_f1:
    print("Better macro F1 on this validation set: expert rules.")
else:
    print("Both methods have the same macro F1 on this validation set.")


Features selected by mutual information:
['e_time', 'i_e_ratio', 'rr', 'tve', 'tve_tvi_ratio', 'maw', 'ip_auc', 'ep_auc']
\nExpert rules — validation classification report:
              precision    recall  f1-score   support

     Non-PVA       0.94      0.94      0.94       842
         BSA       0.86      0.99      0.92       301
         DTA       0.85      0.50      0.63       104

    accuracy                           0.92      1247
   macro avg       0.89      0.81      0.83      1247
weighted avg       0.92      0.92      0.91      1247

Mutual-information + Random Forest — validation classification report:
              precision    recall  f1-score   support

     Non-PVA       0.91      0.96      0.93       842
         BSA       0.84      0.97      0.90       301
         DTA       1.00      0.07      0.13       104

    accuracy                           0.89      1247
   macro avg       0.91      0.67      0.65      1247
weighted avg       0.90      0.89      0.86      

# **4. K-Fold Cross Validation: Systematically evaluating your model**

K-fold cross-validation is a resampling procedure used to systematically evaluate machine learning models. The procedure has a parameter called k that refers to the number of groups that a given data sample is to be split into. This technique is important in machine learning because it ensures that every observation from the original dataset has the chance of appearing in the training and test set, providing a thorough assessment of how well a model performs across different subsets of data.

In [ ]:
from sklearn.model_selection import StratifiedKFold

##Define k
k = 3

##initialize k fold setup for cross-validation
kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)

# Model
model = RandomForestClassifier()

# Running through folds
for train_index, val_index in kf.split(train_x[columns_to_use], train_y_vector):

      X_train, X_test = train_x[columns_to_use].iloc[train_index], train_x[columns_to_use].iloc[val_index]
      y_train, y_test = train_y_vector[train_index], train_y_vector[val_index]

      # Now we can fit the model, generate predictions, and compare the performance with validation labels y_val.
      # Once that is done, we save the results for next fold/iteration

## **Task \#5: Perform k-fold cross-validation** (30 points)

Evaluate an ML model of your choosing (Logistic Regression, etc...) under 5-fold cross-validation. Print the macro average precision, recall, and f1-scores in each fold. Print the average values across all 5 folds at the end of the loop. If you prefer, you can use sklearn's `precision_score()`, `recall_score()`, `f1_score()` instead of `classification_report()`. Please use random state 42.

In [27]:
# Task 5: Five-fold stratified cross-validation (random state 42)

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np
import pandas as pd

cv_X = train_x[feature_columns].astype(float).reset_index(drop=True)
cv_y = train_y_vector.astype(int).reset_index(drop=True)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_results = []

for fold, (fit_indices, val_indices) in enumerate(skf.split(cv_X, cv_y), start=1):
    X_fit, X_val = cv_X.iloc[fit_indices], cv_X.iloc[val_indices]
    y_fit, y_val = cv_y.iloc[fit_indices], cv_y.iloc[val_indices]

    cv_model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            max_iter=3000, random_state=42
        ))
    ])

    cv_model.fit(X_fit, y_fit)
    fold_predictions = cv_model.predict(X_val)

    precision = precision_score(
        y_val, fold_predictions, average="macro", zero_division=0
    )
    recall = recall_score(
        y_val, fold_predictions, average="macro", zero_division=0
    )
    f1 = f1_score(
        y_val, fold_predictions, average="macro", zero_division=0
    )

    fold_results.append({
        "fold": fold,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    })

    print(
        f"Fold {fold}: macro precision={precision:.4f}, "
        f"macro recall={recall:.4f}, macro F1={f1:.4f}"
    )

results_df = pd.DataFrame(fold_results)
print("\\nFold results:")
display(results_df)

print("\\nAverage across all 5 folds:")
print(results_df[["macro_precision", "macro_recall", "macro_f1"]].mean().round(4))


Fold 1: macro precision=0.8619, macro recall=0.7045, macro F1=0.7607
Fold 2: macro precision=0.8681, macro recall=0.7200, macro F1=0.7750
Fold 3: macro precision=0.8442, macro recall=0.6668, macro F1=0.7267
Fold 4: macro precision=0.8681, macro recall=0.6885, macro F1=0.7506
Fold 5: macro precision=0.8544, macro recall=0.6652, macro F1=0.7256
\nFold results:


,fold,macro_precision,macro_recall,macro_f1
0,1,0.861882,0.704468,0.760683
1,2,0.868146,0.719950,0.775011
2,3,0.844151,0.666848,0.726699
3,4,0.868105,0.688492,0.750621
4,5,0.854382,0.665241,0.725555


\nAverage across all 5 folds:
macro_precision    0.8593
macro_recall       0.6890
macro_f1           0.7477
dtype: float64


**Q: In the tutorial, you are taught to use the function StratifiedKFold. What does it mean when we stratify our k-folds?**

In [29]:
print(
    "Stratified k-fold cross-validation keeps approximately the same "
    "proportion of each target class in every fold as in the full dataset. "
    "This is useful for imbalanced classification because each fold is more "
    "likely to contain examples of all classes, making evaluation more reliable."
)


Stratified k-fold cross-validation keeps approximately the same proportion of each target class in every fold as in the full dataset. This is useful for imbalanced classification because each fold is more likely to contain examples of all classes, making evaluation more reliable.


# Report
Check Canvas for a md file containing questions to answer.